In [5]:
#Part A: File Reading & Chat Parser (Feature 1)
# AI-assisted: Starter structure for edge-case WhatsApp parsing without regex
import numpy as np
from datetime import datetime, timedelta

def starts_with_date(line):
    """Checks 'DD/MM/YY' pattern at the start of a line without using re."""
    if len(line) < 8:
        return False
    return (
        line[0:2].isdigit() and line[2] == '/' and
        line[3:5].isdigit() and line[5] == '/' and
        line[6:8].isdigit()
    )

def parse_whatsapp_chat(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_lines = f.read().split('\n')

    messages = []
    system_count = 0
    media_per_person = {}
    deleted_per_person = {}

    for line in raw_lines:
        if not line.strip():
            continue  # Edge case e: skip empty lines silently

        if starts_with_date(line):
            # Format: "DD/MM/YY, HH:MM <Sender>: <Message>"
            ts_str = line[:15]
            remainder = line[15:].strip()
            if remainder.startswith('- '):  # Also supports real Android exports
                remainder = remainder[2:]

            if ':' not in remainder:
                system_count += 1  # Edge case a: system message
                continue

            sender, text = remainder.split(':', 1)
            sender = sender.strip()
            text = text.strip()

            if text == '<Media omitted>':
                media_per_person[sender] = media_per_person.get(sender, 0) + 1
                continue  # Edge case b: exclude from word/length analysis

            if text == 'This message was deleted':
                deleted_per_person[sender] = deleted_per_person.get(sender, 0) + 1
                continue  # Edge case c: track deleted messages separately

            messages.append({
                'timestamp': ts_str,
                'dt': datetime.strptime(ts_str, '%d/%m/%y, %H:%M'),
                'sender': sender,
                'text': text
            })
        else:
            # Edge case d: multi-line message continuation
            if messages:
                messages[-1]['text'] += ' ' + line.strip()

    total_media = sum(media_per_person.values())
    total_deleted = sum(deleted_per_person.values())
    participants = sorted({m['sender'] for m in messages})
    days = (messages[-1]['dt'].date() - messages[0]['dt'].date()).days + 1

    print(
        f"Successfully parsed {len(messages)} messages from {len(participants)} "
        f"participants over {days} days, skipped {system_count} system messages, "
        f"{total_media} media-omitted, {total_deleted} deleted messages."
    )
    return messages, participants, media_per_person, deleted_per_person

    # Call the function using the Google Colab file path
file_path = '/content/hostel_bois.txt'  # Use 'hostel_bois.txt' if running locally
messages, participants, media_counts, deleted_counts = parse_whatsapp_chat(file_path)

# Day 1 Checkpoint: Verify the first 5 and last 5 parsed messages
print("\nFirst 5 messages:")
for msg in messages[:5]:
    print(msg)

print("\nLast 5 messages:")
for msg in messages[-5:]:
    print(msg)

Successfully parsed 3127 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.

First 5 messages:
{'timestamp': '01/04/24, 01:17', 'dt': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'dt': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'dt': datetime.datetime(2024, 4, 1, 1, 18), 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'dt': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'dt': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'text': 'abey'}

Last 5 messages:
{'timestamp': '30/05/24, 19:14', 'dt': datetime.datetime(2024, 5, 30, 19, 14), 'sender': 'Priya', 'text': 'Take care everyone'}
{'timestamp': '30/05/24, 19:28', 'dt': datetime.datetime(2024, 5, 30, 19, 28), 'sender': 'Priya', 'text': 'Karan that sou

In [6]:
#Part B: Overview, Peak Activity & Word Frequency (Features 2, 3 & 5)
def analyze_overview_and_words(messages):
    msg_counts = {}
    day_counts = {}
    hour_counts = {}
    word_counts = {}

    stop_words = {
        'i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on',
        'for', 'it', 'you', 'me', 'my', 'we', 'at', 'be', 'this', 'that',
        'are', 'was', 'with', 'so', 'but', 'have', 'not', 'hai', 'ka', 'ki', 'ke'
    }
    punctuation = '.,!?":;()[]{}<>-_/\\\'*'

    for m in messages:
        sender = m['sender']
        dt = m['dt']
        msg_counts[sender] = msg_counts.get(sender, 0) + 1

        day_label = dt.strftime('%d %B %Y')
        day_counts[day_label] = day_counts.get(day_label, 0) + 1
        hour_counts[dt.hour] = hour_counts.get(dt.hour, 0) + 1

        for raw_word in m['text'].lower().split():
            clean_word = raw_word.strip(punctuation)
            if clean_word and clean_word not in stop_words:
                word_counts[clean_word] = word_counts.get(clean_word, 0) + 1

    busiest_day = max(day_counts, key=day_counts.get)
    busiest_hour = max(hour_counts, key=hour_counts.get)
    top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

    return msg_counts, busiest_day, day_counts[busiest_day], busiest_hour, top_words

    # Run Part B (Features 2, 3 & 5)
msg_counts, busiest_day, busiest_day_count, busiest_hour, top_words = analyze_overview_and_words(messages)

print(f"Busiest Day : {busiest_day} ({busiest_day_count} messages)")
print(f"Busiest Hour: {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00")

print("\nMessages Per Person:")
for person, count in sorted(msg_counts.items(), key=lambda x: x[1], reverse=True):
    pct = (count / len(messages)) * 100
    print(f"  {person:<10}: {count} ({pct:.1f}%)")

print("\nTop 10 Words:")
for word, count in top_words:
    bar = "█" * int((count / top_words[0][1]) * 20)
    print(f"  {word:<12}: {count:>4}  {bar}")

Busiest Day : 04 May 2024 (74 messages)
Busiest Hour: 18:00 - 19:00

Messages Per Person:
  Rahul     : 940 (30.1%)
  Priya     : 712 (22.8%)
  Neha      : 624 (20.0%)
  Aman      : 484 (15.5%)
  Karan     : 345 (11.0%)
  Vikas     : 22 (0.7%)

Top 10 Words:
  how         :  321  ████████████████████
  guys        :  318  ███████████████████
  about       :  274  █████████████████
  am          :  260  ████████████████
  today       :  257  ████████████████
  he          :  220  █████████████
  his         :  217  █████████████
  just        :  208  ████████████
  which       :  202  ████████████
  everyone    :  187  ███████████


In [7]:
#Part C: NumPy $6 \times 24$ Activity Heatmap (Feature 4)
# AI-assisted: NumPy matrix creation and row-normalized block rendering
def build_and_print_heatmap(messages, participants):
    p_index = {name: idx for idx, name in enumerate(participants)}
    heatmap = np.zeros((len(participants), 24), dtype=int)

    for m in messages:
        r = p_index[m['sender']]
        c = m['dt'].hour
        heatmap[r, c] += 1

    shades = ['.', '░', '▒', '█']
    print("\nACTIVITY HEATMAP (messages by hour 00 to 23)")
    header = " " * 10 + " ".join(f"{h:02d}" for h in range(24))
    print(header)

    for idx, name in enumerate(participants):
        row = heatmap[idx]
        row_max = np.max(row)
        blocks = []
        for val in row:
            ratio = (val / row_max) if row_max > 0 else 0
            if ratio <= 0.25:
                blocks.append(shades[0])
            elif ratio <= 0.50:
                blocks.append(shades[1])
            elif ratio <= 0.75:
                blocks.append(shades[2])
            else:
                blocks.append(shades[3])
        print(f"{name:<10}" + "  ".join(blocks))

    return heatmap

    # Run Part C (Feature 4)
heatmap = build_and_print_heatmap(messages, participants)


ACTIVITY HEATMAP (messages by hour 00 to 23)
          00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman      ▒  █  █  ▒  █  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  ▒
Karan     .  .  .  .  .  .  .  .  ░  ░  ▒  ░  █  ▒  █  ▒  ▒  ▒  ▒  █  ▒  ░  .  .
Neha      .  .  .  .  .  ░  .  .  ▒  █  █  ░  ▒  ▒  ░  .  ▒  █  █  █  ▒  ░  ░  ░
Priya     .  .  .  .  .  .  .  ░  ▒  █  █  █  █  ▒  ▒  ░  ░  ▒  ▒  █  ▒  ░  ░  .
Rahul     .  .  .  .  .  .  .  .  .  .  .  .  ▒  ░  ░  ▒  ▒  ░  █  ▒  ░  █  ▒  ▒
Vikas     .  .  .  .  .  .  .  ░  ▒  ░  ░  .  ░  ▒  .  ░  ░  █  ▒  ▒  ░  ░  ░  ▒


In [12]:
#Part D: Response Speed & Silent Streaks (Feature 6)
# AI-assisted: Reply gap calculation and consecutive silent day streak logic
def format_duration(seconds):
    if seconds == 0:
        return "0s"
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = int(seconds % 60)

    parts = []
    if hours > 0:
        parts.append(f"{hours}h")
    if minutes > 0:
        parts.append(f"{minutes}m")
    if remaining_seconds > 0 or not parts:
        parts.append(f"{remaining_seconds}s")
    return " ".join(parts)

def analyze_timing_and_streaks(messages, participants):
    reply_gaps = {p: [] for p in participants}
    active_dates = {p: set() for p in participants}

    for i, m in enumerate(messages):
        sender = m['sender']
        active_dates[sender].add(m['dt'].date())

        if i > 0 and messages[i - 1]['sender'] != sender:
            gap_seconds = (m['dt'] - messages[i - 1]['dt']).total_seconds()
            reply_gaps[sender].append(gap_seconds)

    avg_reply_sec = {
        p: (sum(gaps) / len(gaps) if gaps else 0)
        for p, gaps in reply_gaps.items()
    }

    start_date = messages[0]['dt'].date()
    end_date = messages[-1]['dt'].date()
    total_days = (end_date - start_date).days + 1

    silent_streaks = {}
    silent_day_totals = {}

    for p in participants:
        curr_streak = 0
        max_streak = 0
        total_silent = 0
        for d in range(total_days):
            day = start_date + timedelta(days=d)
            if day not in active_dates[p]:
                curr_streak += 1
                total_silent += 1
                if curr_streak > max_streak:
                    max_streak = curr_streak
            else:
                curr_streak = 0
        silent_streaks[p] = max_streak
        silent_day_totals[p] = total_silent

    return avg_reply_sec, silent_streaks, silent_day_totals, total_days

# Run Part D (Feature 6)
avg_reply_sec, silent_streaks, silent_day_totals, total_days = analyze_timing_and_streaks(messages, participants)

fastest = min(avg_reply_sec, key=lambda p: avg_reply_sec[p] if avg_reply_sec[p] > 0 else float('inf'))
slowest = max(avg_reply_sec, key=avg_reply_sec.get)
print(f"Fastest replier: {fastest} ({format_duration(avg_reply_sec[fastest])})")
print(f

Fastest replier: Vikas (34m 54s)
Slowest replier: Aman (54m 54s)

Longest Silent Streaks:
  Vikas     : 11 days silent (Total 44/60 days inactive)
  Aman      : 0 days silent (Total 0/60 days inactive)
  Karan     : 0 days silent (Total 0/60 days inactive)
  Neha      : 0 days silent (Total 0/60 days inactive)
  Priya     : 0 days silent (Total 0/60 days inactive)
  Rahul     : 0 days silent (Total 0/60 days inactive)


In [13]:
#Part E: Archetype Detection & Custom 9th Archetype (Feature 7)
def detect_archetypes(messages, participants, heatmap, silent_day_totals, total_days):
    # Compute consecutive message bursts per person
    bursts = {p: [] for p in participants}
    curr_sender = None
    curr_len = 0
    for m in messages:
        if m['sender'] == curr_sender:
            curr_len += 1
        else:
            if curr_sender is not None:
                bursts[curr_sender].append(curr_len)
            curr_sender = m['sender']
            curr_len = 1
    if curr_sender is not None:
        bursts[curr_sender].append(curr_len)

    caring_words = [
        'okay', 'safe', 'eat', 'sleep', 'take care',
        'are you', 'please', 'reminder', 'drink water', "don't forget"
    ]
    comedy_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

    scores = {}
    for idx, p in enumerate(participants):
        p_msgs = [m['text'] for m in messages if m['sender'] == p]
        n = len(p_msgs) if p_msgs else 1

        avg_burst = sum(bursts[p]) / len(bursts[p]) if bursts[p] else 0
        caring_hits = sum(
            1 for txt in p_msgs if any(cw in txt.lower() for cw in caring_words)
        )
        night_msgs = np.sum(heatmap[idx, [23, 0, 1, 2, 3, 4]])
        night_pct = (night_msgs / n) * 100
        avg_words = sum(len(txt.split()) for txt in p_msgs) / n

        drama_msgs = sum(
            1 for txt in p_msgs
            if (len(txt) >= 3 and txt.isupper()) or txt.count('!') >= 2
        )
        drama_pct = (drama_msgs / n) * 100
        silent_pct = (silent_day_totals[p] / total_days) * 100
        comedy_pct = (
            sum(1 for txt in p_msgs if any(w in txt.lower().split() for w in comedy_words)) / n
        ) * 100
        question_pct = (sum(1 for txt in p_msgs if txt.strip().endswith('?')) / n) * 100

        # Normalize into comparable priority/strength scores when thresholds are met
        scores[p] = {
            'THE SPAMMER': (avg_burst * 20) if avg_burst > 3 else 0,
            'THE GROUP MOM': caring_hits,
            'THE NIGHT OWL': night_pct if night_pct > 60 else 0,
            'THE STORYTELLER': avg_words if avg_words > 30 else 0,
            'THE DRAMA QUEEN': drama_pct if drama_pct > 30 else 0,
            'THE GHOST': silent_pct if silent_pct > 60 else 0,
            'THE COMEDIAN': comedy_pct,
            'THE QUESTION MASTER': question_pct if question_pct > 25 else 0,
            # TODO: Add your own 9th archetype here for +2 bonus marks!
        }

    # Exclusive assignment: sort all (person, archetype, score) pairs descending
    candidates = []
    for p, arch_dict in scores.items():
        for arch, sc in arch_dict.items():
            candidates.append((sc, p, arch))
    candidates.sort(reverse=True)

    assigned = {}
    taken_archetypes = set()
    for sc, p, arch in candidates:
        if p not in assigned and arch not in taken_archetypes:
            assigned[p] = (arch, sc)
            taken_archetypes.add(arch)

    return assigned

    # AI-assisted: Archetype scoring with reason strings and bonus 9th archetype
def detect_archetypes(messages, participants, heatmap, silent_day_totals, total_days):
    bursts = {p: [] for p in participants}
    curr_sender = None
    curr_len = 0
    for m in messages:
        if m['sender'] == curr_sender:
            curr_len += 1
        else:
            if curr_sender is not None:
                bursts[curr_sender].append(curr_len)
            curr_sender = m['sender']
            curr_len = 1
    if curr_sender is not None:
        bursts[curr_sender].append(curr_len)

    caring_words = [
        'okay', 'safe', 'eat', 'sleep', 'take care',
        'are you', 'please', 'reminder', 'drink water', "don't forget"
    ]
    comedy_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    # Custom 9th archetype keywords (The Scene Maker - Indian college hangout planner)
    plan_words = ['scene', 'chalte', 'plan', 'aaja', 'nikalte', 'ready', 'kahan']

    scores = {}
    stat_labels = {}

    for idx, p in enumerate(participants):
        p_msgs = [m['text'] for m in messages if m['sender'] == p]
        n = len(p_msgs) if p_msgs else 1

        avg_burst = sum(bursts[p]) / len(bursts[p]) if bursts[p] else 0
        caring_hits = sum(1 for txt in p_msgs if any(cw in txt.lower() for cw in caring_words))
        night_msgs = np.sum(heatmap[idx, [23, 0, 1, 2, 3, 4]])
        night_pct = (night_msgs / n) * 100
        avg_words = sum(len(txt.split()) for txt in p_msgs) / n

        drama_msgs = sum(
            1 for txt in p_msgs
            if (len(txt) >= 3 and txt.isupper()) or txt.count('!') >= 2
        )
        drama_pct = (drama_msgs / n) * 100
        silent_pct = (silent_day_totals[p] / total_days) * 100
        comedy_pct = (sum(1 for txt in p_msgs if any(w in txt.lower().split() for w in comedy_words)) / n) * 100
        question_pct = (sum(1 for txt in p_msgs if txt.strip().endswith('?')) / n) * 100
        plan_pct = (sum(1 for txt in p_msgs if any(w in txt.lower().split() for w in plan_words)) / n) * 100

        scores[p] = {
            'THE SPAMMER': (avg_burst * 20) if avg_burst > 3 else 0,
            'THE GROUP MOM': caring_hits,
            'THE NIGHT OWL': night_pct if night_pct > 60 else 0,
            'THE STORYTELLER': avg_words if avg_words > 30 else 0,
            'THE DRAMA QUEEN': drama_pct if drama_pct > 30 else 0,
            'THE GHOST': silent_pct if silent_pct > 60 else 0,
            'THE COMEDIAN': comedy_pct,
            'THE QUESTION MASTER': question_pct if question_pct > 25 else 0,
            # Bonus 9th archetype: defended threshold > 25% so it doesn't override core dataset characters
            'THE SCENE MAKER': plan_pct if plan_pct > 25 else 0,
        }

        stat_labels[p] = {
            'THE SPAMMER': f"avg {avg_burst:.1f} msgs in a row",
            'THE GROUP MOM': f"caring keyword score: {caring_hits}",
            'THE NIGHT OWL': f"{night_pct:.1f}% msgs between 23h-04h",
            'THE STORYTELLER': f"avg {avg_words:.1f} words per msg",
            'THE DRAMA QUEEN': f"{drama_pct:.1f}% ALL-CAPS / dramatic msgs",
            'THE GHOST': f"silent on {silent_day_totals[p]} of {total_days} days",
            'THE COMEDIAN': f"{comedy_pct:.1f}% comedy keywords",
            'THE QUESTION MASTER': f"{question_pct:.1f}% questions",
            'THE SCENE MAKER': f"{plan_pct:.1f}% hangout planning msgs",
        }

    candidates = []
    for p, arch_dict in scores.items():
        for arch, sc in arch_dict.items():
            candidates.append((sc, p, arch))
    candidates.sort(reverse=True)

    assigned = {}
    reasons = {}
    taken_archetypes = set()

    for sc, p, arch in candidates:
        if p not in assigned and arch not in taken_archetypes:
            assigned[p] = (arch, sc)
            reasons[p] = stat_labels[p][arch]
            taken_archetypes.add(arch)

    return assigned, reasons

# Run Part E (Feature 7)
assigned_archetypes, archetype_reasons = detect_archetypes(
    messages, participants, heatmap, silent_day_totals, total_days
)

print("PERSONALITY ARCHETYPES:")
for person in participants:
    arch, _ = assigned_archetypes[person]
    print(f"  {person:<10} -> {arch:<20} ({archetype_reasons[person]})")

PERSONALITY ARCHETYPES:
  Aman       -> THE NIGHT OWL        (80.4% msgs between 23h-04h)
  Karan      -> THE STORYTELLER      (avg 57.0 words per msg)
  Neha       -> THE DRAMA QUEEN      (63.3% ALL-CAPS / dramatic msgs)
  Priya      -> THE GROUP MOM        (caring keyword score: 459)
  Rahul      -> THE SPAMMER          (avg 4.5 msgs in a row)
  Vikas      -> THE GHOST            (silent on 44 of 60 days)


In [16]:
def format_duration(seconds):
    """Converts raw seconds into readable minutes or hours."""
    mins = seconds / 60
    if mins < 60:
        return f"avg {mins:.1f} minutes"
    return f"avg {mins / 60:.1f} hours"

def get_streak_details(messages, participants):
    """Returns both longest silent streak length and its (start, end) dates."""
    active_dates = {p: set() for p in participants}
    for m in messages:
        active_dates[m['sender']].add(m['dt'].date())

    start_date = messages[0]['dt'].date()
    end_date = messages[-1]['dt'].date()
    total_days = (end_date - start_date).days + 1

    streak_info = {}
    for p in participants:
        best_len = 0
        best_range = None
        curr_len = 0
        curr_start = None

        for d in range(total_days):
            day = start_date + timedelta(days=d)
            if day not in active_dates[p]:
                if curr_len == 0:
                    curr_start = day
                curr_len += 1
                if curr_len > best_len:
                    best_len = curr_len
                    best_range = (curr_start, day)
            else:
                curr_len = 0
                curr_start = None

        streak_info[p] = (best_len, best_range)
    return streak_info

    # Run Feature 8: Complete GroupDNA Dashboard
if not messages:
    print("No valid messages found in the file.")
else:
    print_final_report(
        group_name="Hostel Bois 4ever",
        messages=messages,
        participants=participants,
        msg_counts=msg_counts,
        busiest_day=busiest_day,
        busiest_day_count=busiest_day_count,
        busiest_hour=busiest_hour,
        heatmap=heatmap,
        top_words=top_words,
        avg_reply_sec=avg_reply_sec,
        streak_info=streak_info,
        assigned_archetypes=assigned_archetypes,
        archetype_reasons=archetype_reasons
    )

        GROUP DNA DASHBOARD: Hostel Bois 4ever        
Total Messages Analyzed: 3127
Total Participants     : 6
Busiest Day            : 04 May 2024 (74 messages)
Busiest Hour           : 18:00 - 19:00

MESSAGES PER PERSON:
  Rahul     :  940 (30.1%)
  Priya     :  712 (22.8%)
  Neha      :  624 (20.0%)
  Aman      :  484 (15.5%)
  Karan     :  345 (11.0%)
  Vikas     :   22 (0.7%)

TOP WORDS USED:
  how         :  321
  guys        :  318
  about       :  274
  am          :  260
  today       :  257

RESPONSE TIMES:
  Vikas     : avg 34.9 minutes
  Rahul     : avg 35.2 minutes
  Karan     : avg 36.8 minutes
  Neha      : avg 41.3 minutes
  Priya     : avg 42.6 minutes
  Aman      : avg 54.9 minutes

LONGEST SILENT STREAKS:
  Vikas     : 11 days (23 Apr to 03 May)
  Aman      : 0 days (never went silent)
  Karan     : 0 days (never went silent)
  Neha      : 0 days (never went silent)
  Priya     : 0 days (never went silent)
  Rahul     : 0 days (never went silent)

PERSONALITY ARCHET